<a href="https://colab.research.google.com/github/sam81005/ANN_Digit_Recognizer/blob/main/Speech_Emotion_Recognition_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Speech Emotion Recognition in Google Colab

This notebook runs the full mini project in Colab:

- installs dependencies
- uploads and extracts the RAVDESS dataset zip
- extracts MFCC, Chroma, and Mel Spectrogram features
- trains an SVM classifier
- evaluates the model
- predicts emotion for a new uploaded audio file
- launches a live Gradio demo for microphone or file input


In [ ]:
!pip -q install librosa scikit-learn soundfile joblib gradio

## 1. Upload the RAVDESS Zip File

Run the next cell and upload your RAVDESS `.zip` file.

If you already extracted the dataset in Colab or Google Drive, you can skip this upload cell and set `DATA_DIR` manually in the following cell.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile

DATA_DIR = Path('/content/data/ravdess')
DATA_DIR.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
zip_files = [name for name in uploaded if name.lower().endswith('.zip')]

if zip_files:
    zip_path = Path('/content') / zip_files[0]
    with zipfile.ZipFile(zip_path, 'r') as archive:
        archive.extractall(DATA_DIR)
    print(f'Extracted dataset to: {DATA_DIR}')
else:
    print('No zip uploaded. If your dataset already exists somewhere else, set DATA_DIR manually in the next cell.')

Saving ravdess_demo.zip to ravdess_demo (1).zip
Extracted dataset to: /content/data/ravdess


In [ ]:
DATA_DIR

PosixPath('/content/data/ravdess')

## Optional: Generate Demo Dataset in Colab

If you do not want to upload a dataset zip right now, run the next cell.
It creates a small synthetic RAVDESS-style dataset so the notebook can run end-to-end.

In [ ]:
import numpy as np
import soundfile as sf

def generate_demo_dataset(data_dir: Path) -> None:
    emotion_freqs = {
        '01': 220.0,
        '02': 246.0,
        '03': 262.0,
        '04': 294.0,
        '05': 330.0,
        '06': 349.0,
        '07': 392.0,
        '08': 440.0,
    }
    sample_rate = 22050
    duration_seconds = 1.2
    samples_per_emotion = 5
    data_dir.mkdir(parents=True, exist_ok=True)

    def synthesize_signal(base_freq: float, variant: int) -> np.ndarray:
        num_samples = int(sample_rate * duration_seconds)
        timeline = np.linspace(0, duration_seconds, num_samples, endpoint=False)
        rng = np.random.default_rng(1000 + variant)
        detune = 1.0 + (variant - 2) * 0.015
        phase = variant * 0.3
        signal = 0.24 * np.sin(2 * np.pi * base_freq * detune * timeline + phase)
        signal += 0.08 * np.sin(2 * np.pi * (base_freq * 2.0) * timeline)
        signal += 0.03 * np.sin(2 * np.pi * (base_freq / 2.0) * timeline)
        signal *= np.linspace(0.8, 1.0, num_samples)
        signal += 0.008 * rng.normal(size=num_samples)
        return signal.astype(np.float32)

    for emotion_code, base_freq in emotion_freqs.items():
        for sample_idx in range(samples_per_emotion):
            actor_id = sample_idx + 1
            signal = synthesize_signal(base_freq, sample_idx)
            filename = f'03-01-{emotion_code}-01-01-01-{actor_id:02d}.wav'
            sf.write(data_dir / filename, signal, sample_rate)

    print(f'Generated demo dataset in: {data_dir}')

if not list(DATA_DIR.rglob('*.wav')):
    generate_demo_dataset(DATA_DIR)
else:
    print(f'Using existing dataset under: {DATA_DIR}')

Using existing dataset under: /content/data/ravdess


## 2. Define Imports and Helper Functions

In [ ]:
from pathlib import Path
import numpy as np
import librosa
import joblib

from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

EMOTION_MAP = {
    '01': 'neutral',
    '02': 'calm',
    '03': 'happy',
    '04': 'sad',
    '05': 'angry',
    '06': 'fearful',
    '07': 'disgust',
    '08': 'surprised',
}

DEFAULT_SAMPLE_RATE = 22050
N_MFCC = 40
N_MELS = 128


def summarize_feature(matrix: np.ndarray) -> np.ndarray:
    return np.concatenate([matrix.mean(axis=1), matrix.std(axis=1)])


def extract_feature_vector(audio_path: Path) -> np.ndarray:
    signal, sample_rate = librosa.load(audio_path, sr=DEFAULT_SAMPLE_RATE)
    if signal.size == 0:
        raise ValueError(f'Audio file is empty: {audio_path}')

    stft = np.abs(librosa.stft(signal))
    mfcc = librosa.feature.mfcc(y=signal, sr=sample_rate, n_mfcc=N_MFCC)
    chroma = librosa.feature.chroma_stft(S=stft, sr=sample_rate)
    mel = librosa.feature.melspectrogram(y=signal, sr=sample_rate, n_mels=N_MELS)
    mel_db = librosa.power_to_db(mel, ref=np.max)

    return np.concatenate([
        summarize_feature(mfcc),
        summarize_feature(chroma),
        summarize_feature(mel_db),
    ]).astype(np.float32)


def emotion_from_ravdess_filename(audio_path: Path) -> str:
    parts = audio_path.stem.split('-')
    if len(parts) < 3:
        raise ValueError(f'Unexpected filename format: {audio_path.name}')
    emotion_code = parts[2]
    if emotion_code not in EMOTION_MAP:
        raise ValueError(f'Unsupported emotion code: {emotion_code}')
    return EMOTION_MAP[emotion_code]


def find_audio_files(data_dir: Path) -> list[Path]:
    wav_files = sorted(data_dir.rglob('*.wav'))
    if wav_files:
        return wav_files
    raise FileNotFoundError(
        f'No .wav files found under: {data_dir}. '
        'Upload the dataset zip, point DATA_DIR to the extracted folder, or run the demo dataset cell.'
    )


def load_dataset(data_dir: Path):
    audio_files = find_audio_files(data_dir)
    features = []
    labels = []
    file_paths = []

    for audio_path in audio_files:
        label = emotion_from_ravdess_filename(audio_path)
        feature_vector = extract_feature_vector(audio_path)
        features.append(feature_vector)
        labels.append(label)
        file_paths.append(audio_path)

    return np.vstack(features), labels, file_paths


## 3. Train the Model

In [ ]:
MODEL_OUT = Path('/content/models/ser_baseline.joblib')
MODEL_OUT.parent.mkdir(parents=True, exist_ok=True)
demo_dataset_used = False

if not list(DATA_DIR.rglob('*.wav')):
    print('No dataset found in DATA_DIR. Generating demo dataset automatically...')
    demo_dataset_used = True
    emotion_freqs = {
        '01': 220.0,
        '02': 246.0,
        '03': 262.0,
        '04': 294.0,
        '05': 330.0,
        '06': 349.0,
        '07': 392.0,
        '08': 440.0,
    }
    sample_rate = 22050
    duration_seconds = 1.2
    samples_per_emotion = 5
    DATA_DIR.mkdir(parents=True, exist_ok=True)

    for emotion_code, base_freq in emotion_freqs.items():
        for sample_idx in range(samples_per_emotion):
            actor_id = sample_idx + 1
            num_samples = int(sample_rate * duration_seconds)
            timeline = np.linspace(0, duration_seconds, num_samples, endpoint=False)
            rng = np.random.default_rng(1000 + sample_idx)
            detune = 1.0 + (sample_idx - 2) * 0.015
            phase = sample_idx * 0.3
            signal = 0.24 * np.sin(2 * np.pi * base_freq * detune * timeline + phase)
            signal += 0.08 * np.sin(2 * np.pi * (base_freq * 2.0) * timeline)
            signal += 0.03 * np.sin(2 * np.pi * (base_freq / 2.0) * timeline)
            signal *= np.linspace(0.8, 1.0, num_samples)
            signal += 0.008 * rng.normal(size=num_samples)
            filename = f'03-01-{emotion_code}-01-01-01-{actor_id:02d}.wav'
            sf.write(DATA_DIR / filename, signal.astype(np.float32), sample_rate)
    print(f'Generated demo dataset in: {DATA_DIR}')

features, labels, file_paths = load_dataset(DATA_DIR)
print(f'Loaded {len(labels)} files across {len(set(labels))} emotions.')

x_train, x_test, y_train, y_test, train_files, test_files = train_test_split(
    features,
    labels,
    file_paths,
    test_size=0.2,
    random_state=42,
    stratify=labels,
)

model = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', SVC(kernel='rbf', probability=True, class_weight='balanced')),
])

model.fit(x_train, y_train)
predictions = model.predict(x_test)
print(classification_report(y_test, predictions, digits=4, zero_division=0))

payload = {
    'model': model,
    'labels': sorted(set(labels)),
    'train_files': [str(path) for path in train_files],
    'test_files': [str(path) for path in test_files],
    'demo_dataset_used': demo_dataset_used,
    'dataset_path': str(DATA_DIR),
}
joblib.dump(payload, MODEL_OUT)
print(f'Saved model to: {MODEL_OUT}')
if demo_dataset_used:
    print('Warning: the model was trained on synthetic demo audio, so live microphone predictions will not be reliable.')

Loaded 40 files across 8 emotions.
              precision    recall  f1-score   support

       angry     0.5000    1.0000    0.6667         1
        calm     0.0000    0.0000    0.0000         1
     disgust     1.0000    1.0000    1.0000         1
     fearful     0.0000    0.0000    0.0000         1
       happy     0.0000    0.0000    0.0000         1
     neutral     1.0000    1.0000    1.0000         1
         sad     1.0000    1.0000    1.0000         1
   surprised     1.0000    1.0000    1.0000         1

    accuracy                         0.6250         8
   macro avg     0.5625    0.6250    0.5833         8
weighted avg     0.5625    0.6250    0.5833         8

Saved model to: /content/models/ser_baseline.joblib


## 4. Launch Live Demo

Run the next cell after training to open a stylized sci-fi noir interface where you can record speech from the microphone or upload an audio file.

If the UI does not render inline in Colab, use the public Gradio link printed by the cell output.

In [ ]:
import gradio as gr

payload = joblib.load(MODEL_OUT)
model = payload['model']
demo_dataset_used = payload.get('demo_dataset_used', False)

def predict_emotion_live(audio_path):
    if audio_path is None:
        return 'Awaiting signal...', 'Upload or record speech to begin analysis.', None

    signal, _ = librosa.load(audio_path, sr=DEFAULT_SAMPLE_RATE)
    rms = float(np.sqrt(np.mean(signal ** 2))) if signal.size else 0.0
    if rms < 0.01:
        return 'UNCERTAIN', 'Signal too quiet. Try speaking louder or moving closer to the mic.', None

    feature_vector = extract_feature_vector(Path(audio_path))
    prediction = model.predict([feature_vector])[0]
    probabilities = model.predict_proba([feature_vector])[0]
    top_score = float(np.max(probabilities))
    scores = {
        label: float(score)
        for label, score in sorted(
            zip(model.classes_, probabilities), key=lambda item: item[1], reverse=True
        )
    }

    if demo_dataset_used:
        status = 'Demo model active: trained on synthetic tones, so real mic predictions are only for UI testing.'
    elif top_score < 0.40:
        status = f'Low confidence ({top_score:.2f}). The audio may be outside the training distribution.'
    else:
        status = f'Live inference complete. Confidence: {top_score:.2f}'

    if demo_dataset_used or top_score < 0.40:
        return 'UNCERTAIN', status, scores

    return prediction.upper(), status, scores

custom_css = '''
:root {
  --bg0: #05070b;
  --bg1: #0b1018;
  --panel: rgba(10, 14, 20, 0.82);
  --line: rgba(246, 201, 69, 0.22);
  --accent: #f6c945;
  --accent-soft: rgba(246, 201, 69, 0.16);
  --cyan: #6dd6ff;
  --text: #eef3f8;
  --muted: #93a4b8;
}

.gradio-container {
  background:
    radial-gradient(circle at top center, rgba(246, 201, 69, 0.12), transparent 26%),
    radial-gradient(circle at bottom right, rgba(109, 214, 255, 0.08), transparent 22%),
    linear-gradient(180deg, var(--bg1), var(--bg0));
  color: var(--text);
  font-family: 'Trebuchet MS', 'Segoe UI', sans-serif;
}

.gradio-container::before {
  content: '';
  position: fixed;
  inset: 0;
  pointer-events: none;
  background:
    repeating-linear-gradient(90deg, transparent 0 48px, rgba(255,255,255,0.03) 49px 50px),
    repeating-linear-gradient(180deg, transparent 0 48px, rgba(255,255,255,0.025) 49px 50px);
  opacity: 0.18;
  animation: gridShift 18s linear infinite;
}

@keyframes gridShift {
  0% { transform: translateY(0px); }
  50% { transform: translateY(-12px); }
  100% { transform: translateY(0px); }
}

@keyframes pulseRing {
  0% { transform: scale(0.92); opacity: 0.55; }
  70% { transform: scale(1.08); opacity: 0.12; }
  100% { transform: scale(1.12); opacity: 0; }
}

@keyframes glowSweep {
  0% { transform: translateX(-120%) skewX(-24deg); }
  100% { transform: translateX(220%) skewX(-24deg); }
}

.hero-shell {
  position: relative;
  overflow: hidden;
  border: 1px solid var(--line);
  border-radius: 28px;
  background: linear-gradient(135deg, rgba(15,19,27,0.92), rgba(7,10,14,0.96));
  padding: 28px 30px;
  box-shadow: 0 0 0 1px rgba(255,255,255,0.03), 0 26px 90px rgba(0,0,0,0.42), inset 0 1px 0 rgba(255,255,255,0.04);
}

.hero-shell::after {
  content: '';
  position: absolute;
  top: 0;
  left: -20%;
  width: 30%;
  height: 100%;
  background: linear-gradient(90deg, transparent, rgba(246, 201, 69, 0.16), transparent);
  animation: glowSweep 4.8s linear infinite;
}

.hero-grid {
  display: grid;
  grid-template-columns: 120px 1fr;
  gap: 24px;
  align-items: center;
}

.radar-wrap {
  position: relative;
  width: 108px;
  height: 108px;
  border-radius: 50%;
  border: 1px solid rgba(246, 201, 69, 0.3);
  background: radial-gradient(circle, rgba(246, 201, 69, 0.16), rgba(246, 201, 69, 0.02) 60%, transparent 70%);
  box-shadow: inset 0 0 30px rgba(246, 201, 69, 0.08), 0 0 40px rgba(246, 201, 69, 0.08);
}

.radar-wrap::before,
.radar-wrap::after {
  content: '';
  position: absolute;
  inset: 0;
  border-radius: 50%;
  border: 1px solid rgba(246, 201, 69, 0.28);
  animation: pulseRing 2.8s ease-out infinite;
}

.radar-wrap::after {
  animation-delay: 1.2s;
}

.bat-noir-title {
  margin: 0;
  font-size: 2.2rem;
  letter-spacing: 0.18em;
  color: var(--text);
}

.bat-noir-subtitle {
  margin-top: 10px;
  color: var(--muted);
  max-width: 720px;
  line-height: 1.6;
}

.status-row {
  display: flex;
  gap: 12px;
  flex-wrap: wrap;
  margin-top: 18px;
}

.status-chip {
  padding: 8px 12px;
  border-radius: 999px;
  border: 1px solid rgba(246, 201, 69, 0.18);
  background: rgba(246, 201, 69, 0.08);
  color: var(--text);
  font-size: 0.82rem;
  letter-spacing: 0.08em;
}

.console-panel {
  border: 1px solid var(--line);
  border-radius: 24px;
  background: var(--panel);
  box-shadow: 0 24px 80px rgba(0,0,0,0.3), inset 0 1px 0 rgba(255,255,255,0.04);
}

.console-panel h3 {
  color: var(--accent);
  letter-spacing: 0.12em;
}

.gr-button {
  background: linear-gradient(135deg, #f6c945, #d69f16) !important;
  color: #05070b !important;
  border: none !important;
  border-radius: 999px !important;
  font-weight: 800 !important;
  letter-spacing: 0.08em !important;
  box-shadow: 0 0 0 1px rgba(255,255,255,0.08), 0 12px 30px rgba(246, 201, 69, 0.24) !important;
}

.gr-box, .gr-form, .gradio-container .block {
  border-color: rgba(255,255,255,0.08) !important;
}

.gr-textbox, .gr-audio, .gr-label {
  background: rgba(255,255,255,0.02) !important;
}

@media (max-width: 720px) {
  .hero-grid {
    grid-template-columns: 1fr;
    text-align: center;
  }
  .radar-wrap {
    margin: 0 auto;
  }
}
'''

hero_html = '''
<div class='hero-shell'>
  <div class='hero-grid'>
    <div class='radar-wrap'></div>
    <div>
      <h1 class='bat-noir-title'>NIGHT SIGNAL CONSOLE</h1>
      <div class='bat-noir-subtitle'>A cinematic emotion-detection interface with noir surveillance energy. Record your voice live or upload a speech file and let the model decode the emotional signal.</div>
      <div class='status-row'>
        <div class='status-chip'>SCIFI MODE</div>
        <div class='status-chip'>LIVE MIC READY</div>
        <div class='status-chip'>NOIR VISUALS</div>
        <div class='status-chip'>CONFIDENCE GATE</div>
      </div>
    </div>
  </div>
</div>
'''

with gr.Blocks(theme=gr.themes.Base(), css=custom_css) as demo:
    gr.HTML(hero_html)
    with gr.Row():
        with gr.Column(scale=5, elem_classes=['console-panel']):
            audio_input = gr.Audio(
                sources=['upload', 'microphone'],
                type='filepath',
                label='Speech Input',
            )
            run_button = gr.Button('ANALYZE SIGNAL')
        with gr.Column(scale=4, elem_classes=['console-panel']):
            emotion_output = gr.Textbox(label='Predicted Emotion')
            status_output = gr.Textbox(label='System Status')
            scores_output = gr.Label(label='Class Probabilities')

    run_button.click(
        fn=predict_emotion_live,
        inputs=audio_input,
        outputs=[emotion_output, status_output, scores_output],
    )
    audio_input.change(
        fn=predict_emotion_live,
        inputs=audio_input,
        outputs=[emotion_output, status_output, scores_output],
    )

demo.launch(share=True, inline=False)

/tmp/ipykernel_23446/1426783543.py:232: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Base(), css=custom_css) as demo:
/tmp/ipykernel_23446/1426783543.py:232: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Base(), css=custom_css) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://075107a761b45c47c3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 5. Download the Trained Model (Optional)

In [9]:
from google.colab import files

## 6. Upload a Test Audio File and Predict Emotion (Optional)

In [10]:
uploaded_audio = files.upload()
audio_files = list(uploaded_audio.keys())

if not audio_files:
    raise ValueError('Please upload at least one audio file for prediction.')

audio_path = Path('/content') / audio_files[0]
payload = joblib.load(MODEL_OUT)
model = payload['model']

feature_vector = extract_feature_vector(audio_path)
prediction = model.predict([feature_vector])[0]
probabilities = model.predict_proba([feature_vector])[0]

print(f'Predicted emotion: {prediction}')
print('Class probabilities:')
for label, score in sorted(zip(model.classes_, probabilities), key=lambda item: item[1], reverse=True):
    print(f'  {label:10s} {score:.4f}')

Saving sample_predict.wav to sample_predict.wav
Predicted emotion: angry
Class probabilities:
  fearful    0.2001
  disgust    0.1638
  angry      0.1615
  sad        0.1398
  surprised  0.0993
  neutral    0.0896
  happy      0.0818
  calm       0.0641
